# Healthcare Accessibility Analysis in Northern Ireland

This notebook calculates the distance from Data Zones to the nearest hospital
and estimates the population living more than 20 km from healthcare services.

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import folium

from analysis import (
    load_and_merge_datazones,
    get_hospitals_from_osm,
    clean_hospitals,
    calculate_nearest_hospital_distance,
    calculate_population_far
)

from interactive_map import (
    add_datazones_layer,
    add_hospital_markers,
    add_tooltip_style,
    add_legend,
    add_reset_button,
    add_metric_scale_bar
)

## 1. Load Data Zone Boundaries and Population Data

In [ ]:
# Load Data Zone boundaries and merge with Census population data
dz = load_and_merge_datazones(
    "data/data_zones/DZ2021.shp",
    "data/census/census_2021.xlsx"
)

## 2. Resident Population by Data Zone

This map provides population context for the study area.  These population counts are later used to estimate how many residents live in Data Zones with poor hospital accessibility.

In [ ]:
# Plot resident population by Data Zone to provide context for accessibility analysis
fig, ax = plt.subplots(figsize=(8, 8))

dz.plot(
    column="All usual residents",
    cmap="OrRd",
    legend=True,
    ax=ax,
    edgecolor="white",
    linewidth=0.2,
    legend_kwds={"shrink": 0.7}
)

ax.set_title("Resident Population by Data Zone")
ax.axis("off")

plt.show()

## 3. Retrieve and Clean Hospital Locations

In [ ]:
# Retrieve and clean hospital locations
hospitals = get_hospitals_from_osm(dz)
hospitals = clean_hospitals(hospitals, dz)

In [ ]:
# Plot cleaned hospital locations for visual verification
fig, ax = plt.subplots(figsize=(8, 8))

dz.plot(
    color="#d9d9d9",
    edgecolor="white",
    linewidth=0.2,
    ax=ax
)

hospitals.plot(ax=ax, color="red", markersize=5)

ax.set_title("Cleaned Hospital Locations Across Northern Ireland")
ax.axis("off")

plt.show()

## 4. Calculate Distance to Nearest Hospital

Distance is calculated from a representative point within each Data Zone to the nearest hospital location. This provides a simplified measure of spatial accessibility based on straight-line distance rather than road network travel time.

In [ ]:
# Calculate distance to the nearest hospital for each Data Zone
dz = calculate_nearest_hospital_distance(dz, hospitals)

In [ ]:
# Plot distance to nearest hospital (km) across Northern Ireland
fig, ax = plt.subplots(figsize=(9, 8))

dz.plot(
    column="nearest_hospital_km",
    cmap="viridis",
    legend=True,
    ax=ax,
    edgecolor="none",
    linewidth=0,
    legend_kwds={"shrink": 0.65}
)

ax.set_title("Distance to Nearest Hospital Across Northern Ireland (km)")
ax.axis("off")

plt.show()

### 4.1. Interpretation

Darker colours represent Data Zones located closer to hospital facilities, while greener and yellow tones indicate increasing distance to the nearest hospital.

The map shows clusters of lower-distance Data Zones around hospital locations, with distance generally increasing across more remote and rural parts of Northern Ireland.  Small variations between neighbouring zones reflect the fact that distance is calculated separately for each Data Zone rather than as a continuous surface.

## 5. Identify Populations with Poor Hospital Access

In [ ]:
# Calculate population living beyond the 20 km threshold
dz = calculate_population_far(dz)

In [ ]:
# Calculate the total population living more than 20 km from a hospital
total_population_far = dz["population_far"].sum()

print(f"Total population living more than 20 km from a hospital: {total_population_far:,.0f}")

## 6. Define Accessibility Threshold

A binary accessibility indicator is created to identify Data Zones where the nearest hospital is located more than 20 km away.

This threshold highlights areas of poor geographic access to healthcare services and supports subsequent visualisation.

In [ ]:
# Define accessibility threshold (in km)
threshold_km = 20

# Create binary indicator for poor access
# True = more than 20 km from nearest hospital
# False = within acceptable access distance
dz["affected"] = dz["nearest_hospital_km"] > threshold_km

In [ ]:
# Quick check: count the number of Data Zones beyond the threshold
affected_zones = dz["affected"].sum()
print(f"Number of affected Data Zones: {affected_zones}")

## 7. Visualise Accessibility Outcomes

In [ ]:
# Highlight Data Zones where resident population lives more than 20 km from the nearest hospital
fig, ax = plt.subplots(figsize=(8, 8))

# Base map showing all Data Zones for geographic context
dz.plot(
    color="#bfbfbf",
    edgecolor="white",
    linewidth=0.2,
    ax=ax
)

# Overlay Data Zones with populations affected by poor access (>20 km)
dz[dz["affected"]].plot(
    column="population_far",
    cmap="Reds",
    vmin=0,
    legend=True,
    ax=ax,
    edgecolor="darkred",
    linewidth=0.2,
    alpha=0.9,
    legend_kwds={
        "label": "Affected population",
        "shrink": 0.7
    }
)

ax.set_title("Population Living More Than 20 km from the Nearest Hospital")
ax.axis("off")

plt.show()

### 7.1. Interpretation

This map highlights only those Data Zones where residents live more than 20 km from the nearest hospital.  The concentration of affected populations in the west, south-west and a smaller number of peripheral locations elsewhere indicates a clear spatial unevenness in hospital accessibility across Northern Ireland.

The use of a neutral base layer ensures that unaffected Data Zones remain visible for geographic context, while the red overlay highlights areas where poor accessibility coincides with resident population.

These spatial patterns correspond to an estimated 77,000 residents living more than 20 km from a hospital, highlighting the scale of accessibility inequality across the region.

## 8. Create an Interactive Accessibility Map

An interactive map is created using the `folium` library to support exploration of healthcare accessibility patterns across Northern Ireland.

Users can inspect individual Data Zones and view key contextual and accessibility information, including Data Zone name, Data Zone code, county, local council, distance to the nearest hospital and affected population.

### 8.1. Prepare Contextual Fields for the Interactive Map

Additional geographic context is prepared for the interactive map.  County names are added through a spatial join with county boundaries, while full Data Zone names are formatted for display.  Local Government District names are retained from the existing dataset to provide local council context.

A clean copy of the Data Zone dataset is also created for Folium visualisation.  The `zone_point` geometry used in the distance calculations is removed, as only polygon geometries are required for GeoJSON web mapping.

Finally, the Data Zone and hospital layers are converted to WGS84 (`EPSG:4326`), which is the standard coordinate reference system for interactive web maps.

In [ ]:
# Create a copy of the dataset for mapping
# The 'zone_point' column was used in distance calculations and is not needed for visualisation
dz_folium = dz.drop(columns=["zone_point"], errors="ignore").copy()

# Load county boundaries
counties = gpd.read_file("data/counties/counties.shp").to_crs(dz.crs)

# Create representative points for spatial join
dz_points = dz_folium.copy()
dz_points["geometry"] = dz_points.geometry.representative_point()

# Join county names to Data Zones
dz_counties = gpd.sjoin(
    dz_points,
    counties,
    how="left",
    predicate="within"
)

# Add joined county names and formatted Data Zone names to the dataset
dz_folium["county_name"] = dz_counties["NAME"].str.title().values
dz_folium["data_zone_name"] = dz_folium["DZ2021_nm"].str.replace("_", " ")

# Format fields for display in the interactive map
dz_folium["nearest_hospital_km"] = dz_folium["nearest_hospital_km"].round(2)
dz_folium["population_far"] = dz_folium["population_far"].astype(int)

# Convert Data Zones to WGS84 for web mapping
dz_wgs84 = dz_folium.to_crs(epsg=4326)

# Convert hospitals to WGS84 for web mapping
hospitals_wgs84 = hospitals.to_crs(epsg=4326)

In [ ]:
# Calculate map centre using a projected CRS, then convert to WGS84
dz_projected = dz.to_crs(epsg=29902)
center_point = dz_projected.geometry.union_all().centroid
center_point_wgs84 = gpd.GeoSeries([center_point], crs=29902).to_crs(epsg=4326)
center = [
    center_point_wgs84.iloc[0].y,
    center_point_wgs84.iloc[0].x
]

# Store initial map view (center and zoom) for reset button functionality
initial_center = center
initial_zoom = 8

# Create interactive base map
m = folium.Map(
    location=initial_center,
    zoom_start=initial_zoom,
    tiles="CartoDB positron",
    control_scale=False
)

In [ ]:
# Add Data Zone layer with accessibility styling and tooltip information
add_datazones_layer(m, dz_wgs84)

In [ ]:
# Add hospital locations as interactive markers
add_hospital_markers(m, hospitals_wgs84)

### 8.2. Healthcare Accessibility in Northern Ireland
*Interactive Map of Population Accessibility to Hospitals*

⚠️ Methodological Note:

Distances are calculated as straight-line (Euclidean) distances from each Data Zone’s representative point to the nearest hospital location, identified using a spatial nearest-neighbour approach.

These values indicate geographic proximity rather than actual travel distance or travel time by road.

In [ ]:
add_tooltip_style(m)
add_legend(m)

# Add reset button to return to initial map view
add_reset_button(m, initial_center, initial_zoom)

# Add metric-only scale bar (km only)
add_metric_scale_bar(m)

folium.LayerControl().add_to(m)
m

In [ ]:
# Save interactive map as HTML
m.save("interactive_healthcare_access_map.html")